In [1]:
import os 
os.chdir('../../../../')
os.environ["DPM_TQDM"] = "False"
os.environ["CUDA_VISIBLE_DEVICES"]="1"

!nvidia-smi

Wed Aug 13 23:47:18 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4090        Off |   00000000:19:00.0 Off |                  Off |
| 40%   61C    P8             42W /  450W |      11MiB /  24564MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# -*- coding: utf-8 -*-
import os
import math
import numpy as np
from easydict import EasyDict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm

# ===============================
# Config
# ===============================
config = EasyDict()
config.backbone      = 'DiT'
config.train_pt_dir  = 'samplings/dit/train_4.0/dit_train_4.0_1'
config.valid_pt_dir  = 'samplings/dit/eval1000_4.0/dit_eval1000_4.0_0'
config.batch_size    = 10
config.CFG           = 4.0
config.epochs        = 10
config.val_every     = 100
config.log_dir       = "logs/CFG4.0/0813-3:deltaL1:False,deltaL2:True"

# LR & Scheduler
config.base_lr       = 2e-3
config.total_steps   = 1000        # 전체 학습 스텝
config.warmup_steps  = 50          # 워ーム업 스텝
config.min_lr_ratio  = 0.10        # 코사인 최저 비율 (= base_lr * 0.10)

os.makedirs(config.log_dir, exist_ok=True)

# ===============================
# Model (frozen)
# ===============================
from backbones.dit import DiT
from utils.inception import FIDInception

if config.backbone == 'DiT':
    model = DiT(trainable=True)  # 내부 구현에 맞춰 유지
    model.set_freeze()
device = model.device
print(model)
inception = FIDInception().to(device)

# ===============================
# Dataset / Dataloader
# ===============================
from datasets.pt_dataset import PtDataset

train_dataset = PtDataset(config.train_pt_dir)
valid_dataset = PtDataset(config.valid_pt_dir)
print('len(train_dataset) :', len(train_dataset), 'len(valid_dataset) :', len(valid_dataset))

train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=8,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4,
)

valid_loader = DataLoader(valid_dataset, batch_size=config.batch_size, shuffle=False)
print('dataloaders ready')

# ===============================
# Solver / Optimizer / Scheduler
# ===============================
from solvers.dual.dynamic.gdual_solver_log_usedeltaL import GDual_Solver
from solvers.transforms.loglinear_transform_general import LogLinearTransform
from solvers.param_extractors.gap_extractor import GAP_Extractor

noise_schedule = model.get_noise_schedule()
extractor = GAP_Extractor(input_shape=(4, 32, 32))
transform = LogLinearTransform(gamma_push=True, gamma_max=3, kappa_max=3, kappa_disable=True)
solver = GDual_Solver(
    noise_schedule,
    steps=5,
    transform=transform,
    param_extractor=extractor,
    use_deltaL_1 = False,
    use_deltaL_2 = True,
    skip_type="time_uniform",
    time_learning=True,
    train_mode=True
).to(device)

optimizer = torch.optim.AdamW(solver.parameters(), lr=config.base_lr)

def build_warmup_cosine_scheduler(optimizer, total_steps, warmup_steps, min_lr_ratio=0.1):
    assert 0 <= warmup_steps < total_steps
    def lr_lambda(step: int):
        # step: 0,1,2,...
        if step < warmup_steps:
            return max(1e-8, float(step + 1) / float(max(1, warmup_steps)))
        # cosine decay to min ratio
        progress = (step - warmup_steps) / max(1, (total_steps - warmup_steps))
        progress = min(1.0, max(0.0, progress))
        cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
        return min_lr_ratio + (1.0 - min_lr_ratio) * cosine
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

scheduler = build_warmup_cosine_scheduler(
    optimizer,
    total_steps=config.total_steps,
    warmup_steps=config.warmup_steps,
    min_lr_ratio=config.min_lr_ratio,
)
print('solver/optimizer/scheduler ready')

# ===============================
# Utils
# ===============================

def abort_if_bad(tag, value, step=None):
    v = float(value.detach().cpu()) if isinstance(value, torch.Tensor) else float(value)
    if (not math.isfinite(v)) or (v >= 100.0):
        msg = f"[EARLY-STOP] {tag} loss={v:.6f}" + (f" @ step {step}" if step is not None else "")
        print(msg, flush=True)
        raise RuntimeError(msg)

def save_checkpoint(global_step, save_dir, solver, valid_loss):
    ckpt = {
        "global_step": int(global_step),
        "solver_state_dict": solver.state_dict(),
        "valid_loss": float(valid_loss),
        "config": dict(config),
    }
    os.makedirs(save_dir, exist_ok=True)
    step_path = os.path.join(save_dir, f"step_{global_step:08d}.pt")
    torch.save(ckpt, step_path)
    return step_path

@torch.no_grad()
def get_valid_loss(device, solver):
    solver.eval()
    psnr_losses = []
    inception_losses = []
    pbar = tqdm(valid_loader, leave=False)
    for bi, batch in enumerate(pbar):
        noises = batch['noise'].to(device, non_blocking=True)
        conds  = batch['cond']
        targets= batch['sample'].to(device, non_blocking=True)
        target_features= batch['inception_feature'][:, 0].to(device, non_blocking=True)
        
        model_fn = model.get_model_fn(noise_schedule, pos_conds=conds, guidance_scale=config.CFG)
        with torch.no_grad():
            pred = solver.sample(noises, model_fn)
            loss = psnr_loss = torch.log(F.mse_loss(pred, targets) + 1e-8)
            pred = model.decode_vae(pred, raw_output=True)
            pred = inception(pred)
            inception_loss = F.mse_loss(pred, target_features)
        
        abort_if_bad("valid(batch)", loss)     # ← 즉시 중단

        psnr_losses.append(psnr_loss.item())
        inception_losses.append(inception_loss.item())
        pbar.set_postfix({'val_loss': loss.item()})

    val_psnr_mean = float(np.mean(psnr_losses))
    val_inception_mean = float(np.mean(inception_losses))
    abort_if_bad("valid(mean)", val_inception_mean)      # ← 평균도 한 번 더 점검
    return val_psnr_mean, val_inception_mean

from IPython.display import clear_output
def do_train_loop(device, epoch, writer, solver, optimizer, scheduler, global_step_start=0):
    solver.train()
    pbar = tqdm(train_loader)
    losses = []
    global_step = global_step_start

    for step, batch in enumerate(pbar):
        if global_step >= config.total_steps:
            break

        if global_step > 0 and global_step % config.val_every == 0:
            val_psnr_mean, val_inception_mean = get_valid_loss(device, solver)
            print(f'step : {global_step} valid_psnr_loss : {val_psnr_mean:.6f}')
            print(f'step : {global_step} valid_inception_loss : {val_inception_mean:.6f}')
            writer.add_scalar("valid/psnr_loss", val_psnr_mean, global_step)
            writer.add_scalar("valid/inception_loss", val_inception_mean, global_step)
            save_checkpoint(global_step, config.log_dir, solver, val_inception_mean)

        optimizer.zero_grad(set_to_none=True)
        noises = batch['noise'].to(device, non_blocking=True)
        conds  = batch['cond']
        targets= batch['sample'].to(device, non_blocking=True)
        target_features = batch['inception_feature'][:, 0].to(device, non_blocking=True)

        model_fn = model.get_model_fn(noise_schedule, pos_conds=conds, guidance_scale=config.CFG)

        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            pred = solver.sample(noises, model_fn)
            psnr_loss = torch.log(F.mse_loss(pred, targets) + 1e-8)
            pred = model.decode_vae(pred, raw_output=True)
            pred = inception(pred)
            loss = inception_loss = F.mse_loss(pred, target_features)
            cosine_loss = torch.mean(F.cosine_similarity(pred, target_features))
        
        abort_if_bad("train", loss, global_step)  # ← 즉시 중단

        loss.backward()
        # ---- 2) grad norm 기준 클리핑 + NaN 체크
        grad_norm = torch.nn.utils.clip_grad_norm_(solver.parameters(), 1.0)
        if torch.isnan(grad_norm):
            print(f"[SKIP-STEP] non-finite grad_norm={gn.item():.4e}", flush=True)
            optimizer.zero_grad(set_to_none=True)
            continue

        optimizer.step()
        scheduler.step()

        lr_now = optimizer.param_groups[0]["lr"]
        writer.add_scalar("train/lr", lr_now, global_step)
        writer.add_scalar("train/psnr_loss", psnr_loss.item(), global_step)
        writer.add_scalar("train/inception_loss", inception_loss.item(), global_step)
        writer.add_scalar("train/cosine_loss", cosine_loss.item(), global_step)
        
        losses.append(loss.item())
        pbar.set_postfix({'loss': loss.item(), 'lr': lr_now})
        global_step += 1
        #clear_output()

    return float(np.mean(losses)) if losses else 0.0, global_step


/home/scpark/miniconda3/envs/rbf/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading pipeline components...:   0%|          | 0/3 [00:00<?, ?it/s]An error occurred while trying to fetch /data/huggingface/DiT-XL-2-256/transformer: Error no file named diffusion_pytorch_model.safetensors found in directory /data/huggingface/DiT-XL-2-256/transformer.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
Loading pipeline components...:  33%|███▎      | 1/3 [00:00<00:00,  2.96it/s]An error occurred while trying to fetch /data/huggingface/DiT-XL-2-256/vae: Error no file named diffusion_pytorch_model.safetensors found in directory /data/huggingface/DiT-XL-2-256/vae.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
Loading pipeline co

len(train_dataset) : 10000 len(valid_dataset) : 1000
dataloaders ready
solver/optimizer/scheduler ready


In [3]:
# ===============================
# Train
# ===============================
def main():
    writer = SummaryWriter(config.log_dir)
    print('tensorboard:', config.log_dir)

    global_step = 0
    for epoch in range(config.epochs):
        if global_step >= config.total_steps:
            break
        mean_loss, global_step = do_train_loop(
            device, epoch, writer, solver, optimizer, scheduler, global_step_start=global_step
        )
        print(f'[epoch {epoch}] mean_train_loss={mean_loss:.6f}, global_step={global_step}')

    # 마지막 검증 & 체크포인트
    val = get_valid_loss(device, solver)
    save_checkpoint(global_step, config.log_dir, solver, val)
    writer.add_scalar("valid/loss_final", val, global_step)
    writer.close()
    print('done')

if __name__ == "__main__":
    main()


tensorboard: logs/CFG4.0/0813-3:deltaL1:False,deltaL2:True


  1%|          | 9/1000 [00:11<21:38,  1.31s/it, loss=0.195, lr=0.0004] 


RuntimeError: The size of tensor a (4) must match the size of tensor b (2) at non-singleton dimension 1